In [ ]:
import pandas as pd
from math import radians, cos, sin, asin, sqrt

# Load the data
summary = pd.read_csv("data/tmc_most_recent_summary_data.csv")

# Show what columns we actually have
print("Columns in the file:")
print(summary.columns.tolist())
print()

# Show first 10 rows so you see the format
print("First 10 rows:")
print(summary[["location_name", "latest_count_date", "latitude", "longitude"]].head(10))
print()

# Convert count_date to actual date
summary["latest_count_date"] = pd.to_datetime(summary["latest_count_date"])
summary["count_year"] = summary["latest_count_date"].dt.year

# Filter to recent counts
recent = summary[summary["count_year"] >= 2022].copy()
print(f"Total intersections counted since 2022: {len(recent)}")
print()

# Find the days with the MOST intersections counted
# This tells us: "On which day did the city send out the most counting teams?"
counts_by_day = recent.groupby("latest_count_date").size().sort_values(ascending=False)
print("Top 10 days with the most intersections counted:")
print(counts_by_day.head(20))
print()

# Pick the busiest day
busiest_day = counts_by_day.index[0]
print(
    f"Busiest day: {busiest_day.strftime('%Y-%m-%d')} with {counts_by_day.iloc[0]} intersections"
)
print()

# Show all intersections counted on that busiest day
day_data = recent[recent["latest_count_date"] == busiest_day].copy()
print(f"All intersections counted on {busiest_day.strftime('%Y-%m-%d')}:")
print(day_data[["location_name", "latitude", "longitude"]].to_string())
print()

Columns in the file:
['_id', 'latest_count_id', 'latest_count_date', 'location_name', 'longitude', 'latitude', 'centreline_type', 'centreline_id', 'px', 'count_duration', 'total_vehicle', 'total_bike', 'total_heavy_pct', 'total_pedestrian', 'am_peak_start', 'am_peak_vehicle', 'am_peak_bike', 'am_peak_heavy_pct', 'pm_peak_start', 'pm_peak_vehicle', 'pm_peak_bike', 'pm_peak_heavy_pct', 'n_appr_vehicle', 'n_appr_bike', 'n_appr_heavy_pct', 'e_appr_vehicle', 'e_appr_bike', 'e_appr_heavy_pct', 's_appr_vehicle', 's_appr_bike', 's_appr_heavy_pct', 'w_appr_vehicle', 'w_appr_bike', 'w_appr_heavy_pct']

First 10 rows:
                                       location_name latest_count_date  \
0                        Kennedy Rd / Lawrence Ave E        2026-08-26   
1               Monarch Park Ave / Sammon Ave (West)        2026-08-26   
2  Hwy 401 Collectors E Yonge St N Ramp / Lord Se...        2026-08-25   
3                             Yore Rd / Trethewey Dr        2026-08-25   
4              

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from math import radians
from collections import defaultdict

# Load data
df = pd.read_csv("data/tmc_most_recent_summary_data.csv")
df["latest_count_date"] = pd.to_datetime(df["latest_count_date"])

print(f"Total intersections: {len(df)}")

# Convert lat/lon to radians for haversine-ready tree
# cKDTree can't do haversine directly, so we use a Cartesian approximation
# At Toronto's latitude (~43.7°), 1 degree lat ≈ 111 km, 1 degree lon ≈ 80 km
# We scale lon to match lat, then use Euclidean distance in this scaled space
RADIUS_KM = 0.6
LAT_KM_PER_DEG = 111.0
LON_KM_PER_DEG = 111.0 * np.cos(np.radians(43.7))

coords = np.column_stack(
    [df["latitude"].values * LAT_KM_PER_DEG, df["longitude"].values * LON_KM_PER_DEG]
)

# Build spatial index — O(n log n), query is O(log n) per point
tree = cKDTree(coords)

# Find all pairs within RADIUS_KM
# pairs[i] = [index_a, index_b] where distance < RADIUS_KM
pairs = tree.query_pairs(r=RADIUS_KM, output_type="ndarray")

print(f"Found {len(pairs)} neighbor pairs within {RADIUS_KM} km")

# Build adjacency list: for each intersection, who are its neighbors?

neighbors = defaultdict(set)
for a, b in pairs:
    neighbors[a].add(b)
    neighbors[b].add(a)

# Find clusters: each intersection + its neighbors
MIN_SIZE = 6
MAX_SIZE = 12

clusters = []
seen = set()

for i in range(len(df)):
    if i in seen:
        continue

    # Get this intersection and all its neighbors
    cluster_indices = {i} | neighbors[i]

    # If too small, skip
    if len(cluster_indices) < MIN_SIZE:
        continue

    # If too big, keep only the closest ones
    cluster_indices = list(cluster_indices)
    if len(cluster_indices) > MAX_SIZE:
        # Calculate distances from anchor (i) to all neighbors
        anchor_lat = coords[i][0]
        anchor_lon = coords[i][1]
        dists = []
        for j in cluster_indices:
            d = np.sqrt(
                (coords[j][0] - anchor_lat) ** 2 + (coords[j][1] - anchor_lon) ** 2
            )
            dists.append((d, j))
        dists.sort()
        cluster_indices = [j for _, j in dists[:MAX_SIZE]]

    cluster_indices = set(cluster_indices)

    # Deduplicate: if we've seen any of these before, skip
    if cluster_indices & seen:
        continue

    seen.update(cluster_indices)

    members = df.iloc[list(cluster_indices)].copy()
    members["distance_km"] = np.sqrt(
        (
            (members["latitude"] * LAT_KM_PER_DEG - coords[i][0]) ** 2
            + (members["longitude"] * LON_KM_PER_DEG - coords[i][1]) ** 2
        )
    ).round(3)
    members = members.sort_values("distance_km")

    clusters.append(
        {
            "anchor_name": df.iloc[i]["location_name"],
            "anchor_id": df.iloc[i]["centreline_id"],
            "size": len(members),
            "members": members,
        }
    )

# Sort by size descending
clusters.sort(key=lambda x: x["size"], reverse=True)

print(f"\nFound {len(clusters)} valid clusters\n")

# Display top clusters
for idx, cluster in enumerate(clusters[:15]):
    print(f"{'=' * 70}")
    print(f"CLUSTER #{idx + 1} | Anchor: {cluster['anchor_name']}")
    print(f"Centreline ID: {cluster['anchor_id']} | Size: {cluster['size']}")
    print(f"{'-' * 70}")

    display_cols = [
        "location_name",
        "centreline_id",
        "distance_km",
        "latest_count_date",
        "total_vehicle",
    ]
    print(cluster["members"][display_cols].to_string(index=False))

    dates = cluster["members"]["latest_count_date"].dt.date.unique()
    print(f"\nUnique count dates: {len(dates)} | {sorted([str(d) for d in dates])}")
    print()

# Save all clusters to a file for easy reference
with open("data/cluster_candidates.txt", "w") as f:
    for idx, cluster in enumerate(clusters):
        f.write(
            f"CLUSTER #{idx + 1}: {cluster['anchor_name']} (ID: {cluster['anchor_id']})\n"
        )
        f.write(f"Size: {cluster['size']}\n")
        for _, row in cluster["members"].iterrows():
            f.write(
                f"  - {row['location_name']} (ID: {row['centreline_id']}, "
                f"{row['distance_km']} km, {row['latest_count_date'].date()}, "
                f"{row['total_vehicle']} veh)\n"
            )
        f.write("\n")

print(f"Saved all {len(clusters)} clusters to data/cluster_candidates.txt")

Total intersections: 6390
Found 64286 neighbor pairs within 0.6 km

Found 412 valid clusters

CLUSTER #1 | Anchor: Monarch Park Ave / Sammon Ave (West)
Centreline ID: 13460149 | Size: 12
----------------------------------------------------------------------
                         location_name  centreline_id  distance_km latest_count_date  total_vehicle
  Monarch Park Ave / Sammon Ave (West)       13460149        0.000        2026-08-26           2429
  Monarch Park Ave / Sammon Ave (East)       13460141        0.015        2020-08-18            665
         Bonnie Brae Blvd / Sammon Ave       13460202        0.115        2004-09-15           1406
               Sammon Ave / Durant Ave       13460058        0.154        2020-09-23           1207
Monarch Park Ave / Mortimer Ave (East)       13459830        0.208        1999-07-29           5511
     Sammon Ave / Linsmore Cres (West)       13460252        0.215        2020-09-22            962
Queensdale Ave / Linsmore Cres (South)    

In [2]:
raw = pd.read_csv("data/tmc_raw_data_2020_2029.csv")

# Your 12 centreline_ids from Cluster #15
my_ids = [
    13467140,
    13467056,
    13467253,
    13466953,
    13467321,
    13467106,
    13466978,
    13467173,
    13466829,
    13467365,
    13466887,
    13467525,
]

my_data = raw[raw["centreline_id"].isin(my_ids)]

# How many count dates does EACH intersection have?
dates_per_intersection = (
    my_data.groupby("centreline_id")["count_date"]
    .nunique()
    .sort_values(ascending=False)
)
print("Number of count dates per intersection:")
print(dates_per_intersection)

# Show all dates for each intersection
print("\nAll count dates:")
for cid in my_ids:
    name = my_data[my_data["centreline_id"] == cid]["location_name"].iloc[0]
    dates = sorted(my_data[my_data["centreline_id"] == cid]["count_date"].unique())
    print(f"{name} (ID: {cid}): {len(dates)} dates — {dates}")


Number of count dates per intersection:
centreline_id
13466829    1
13466887    1
13466953    1
13466978    1
13467056    1
13467106    1
13467140    1
13467173    1
13467253    1
13467321    1
13467525    1
Name: count_date, dtype: int64

All count dates:
Pearl St / Duncan St / Ed Mirvish Way (ID: 13467140): 1 dates — ['2026-08-11']
Adelaide St W / Duncan St (ID: 13467056): 1 dates — ['2025-09-10']
King St W / Ed Mirvish Way (ID: 13467253): 1 dates — ['2026-08-11']
Duncan St / Nelson St (ID: 13466953): 1 dates — ['2026-08-11']
King St W / John St (ID: 13467321): 1 dates — ['2025-09-10']
Adelaide St W / John St (ID: 13467106): 1 dates — ['2025-09-10']
Adelaide St W / Simcoe St (ID: 13466978): 1 dates — ['2025-10-15']
King St W / Simcoe St (ID: 13467173): 1 dates — ['2025-10-15']
Richmond St W / Duncan St (ID: 13466829): 1 dates — ['2025-09-09']


IndexError: single positional indexer is out-of-bounds

In [3]:
import pandas as pd
from math import radians, cos, sin, asin, sqrt
import numpy as np
from scipy.spatial import cKDTree


def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return c * 6371


# Load summary and raw
summary = pd.read_csv("data/tmc_most_recent_summary_data.csv")
raw = pd.read_csv("data/tmc_raw_data_2020_2029.csv")

# Ensure centreline_id is string in both (sometimes raw has ints, summary has strings)
summary["centreline_id"] = summary["centreline_id"].astype(str)
raw["centreline_id"] = raw["centreline_id"].astype(str)
raw["count_date"] = pd.to_datetime(raw["count_date"])

print(f"Summary file: {len(summary)} intersections")
print(
    f"Raw file: {len(raw)} rows, {raw['centreline_id'].nunique()} unique intersections"
)
print()

# Pre-compute: how many dates does each centreline_id have in raw?
dates_per_id = raw.groupby("centreline_id")["count_date"].nunique().to_dict()

# Also: what are the actual dates for each id?
all_dates_per_id = (
    raw.groupby("centreline_id")["count_date"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .to_dict()
)

# Build clusters (same logic as before)
RADIUS_KM = 0.6
LAT_KM_PER_DEG = 111.0
LON_KM_PER_DEG = 111.0 * np.cos(np.radians(43.7))

coords = np.column_stack(
    [
        summary["latitude"].values * LAT_KM_PER_DEG,
        summary["longitude"].values * LON_KM_PER_DEG,
    ]
)

tree = cKDTree(coords)
pairs = tree.query_pairs(r=RADIUS_KM, output_type="ndarray")

from collections import defaultdict

neighbors = defaultdict(set)
for a, b in pairs:
    neighbors[a].add(b)
    neighbors[b].add(a)

MIN_SIZE = 6
MAX_SIZE = 12

clusters = []
seen = set()

for i in range(len(summary)):
    if i in seen:
        continue
    cluster_indices = {i} | neighbors[i]
    if len(cluster_indices) < MIN_SIZE:
        continue
    cluster_indices = list(cluster_indices)
    if len(cluster_indices) > MAX_SIZE:
        anchor_lat, anchor_lon = coords[i]
        dists = []
        for j in cluster_indices:
            d = np.sqrt(
                (coords[j][0] - anchor_lat) ** 2 + (coords[j][1] - anchor_lon) ** 2
            )
            dists.append((d, j))
        dists.sort()
        cluster_indices = [j for _, j in dists[:MAX_SIZE]]
    cluster_indices = set(cluster_indices)
    if cluster_indices & seen:
        continue
    seen.update(cluster_indices)

    members = summary.iloc[list(cluster_indices)].copy()
    members["distance_km"] = np.sqrt(
        (
            (members["latitude"] * LAT_KM_PER_DEG - coords[i][0]) ** 2
            + (members["longitude"] * LON_KM_PER_DEG - coords[i][1]) ** 2
        )
    ).round(3)
    members = members.sort_values("distance_km")

    clusters.append(
        {"anchor_name": summary.iloc[i]["location_name"], "members": members}
    )

clusters.sort(key=lambda x: len(x["members"]), reverse=True)

print(f"Built {len(clusters)} clusters. Now checking raw data coverage...\n")

# Score each cluster
scored_clusters = []

for idx, cluster in enumerate(clusters):
    members = cluster["members"]
    cids = members["centreline_id"].tolist()

    # Check which IDs exist in raw data
    found_in_raw = [cid for cid in cids if cid in dates_per_id]
    missing_from_raw = [cid for cid in cids if cid not in dates_per_id]

    if len(found_in_raw) < 6:
        continue  # Need at least 6 intersections with raw data

    # Count dates per found intersection
    date_counts = {cid: dates_per_id[cid] for cid in found_in_raw}
    multi_date_ids = [cid for cid, count in date_counts.items() if count >= 2]

    # Find shared dates: which dates appear for multiple intersections?
    date_intersections = defaultdict(list)
    for cid in found_in_raw:
        for d in all_dates_per_id[cid]:
            date_intersections[d].append(cid)

    # Find the best pair of dates for calibration/validation
    # We want two different dates, each with the most intersections covered
    shared_dates = sorted(
        date_intersections.items(), key=lambda x: len(x[1]), reverse=True
    )

    best_cal_date = None
    best_val_date = None
    best_coverage = 0

    for cal_date, cal_ids in shared_dates:
        for val_date, val_ids in shared_dates:
            if cal_date == val_date:
                continue
            # Intersections that have BOTH dates
            both = set(cal_ids) & set(val_ids)
            if len(both) > best_coverage:
                best_coverage = len(both)
                best_cal_date = cal_date
                best_val_date = val_date

    score = {
        "cluster_num": idx + 1,
        "anchor": cluster["anchor_name"],
        "size": len(members),
        "found_in_raw": len(found_in_raw),
        "missing_from_raw": len(missing_from_raw),
        "multi_date_count": len(multi_date_ids),
        "best_cal_date": best_cal_date,
        "best_val_date": best_val_date,
        "best_coverage": best_coverage,  # Intersections with BOTH dates
        "members": members,
        "date_counts": date_counts,
        "shared_dates": shared_dates[:5],  # Top 5 shared dates
    }
    scored_clusters.append(score)

# Sort by: (1) best_coverage desc, (2) multi_date_count desc, (3) size desc
scored_clusters.sort(
    key=lambda x: (x["best_coverage"], x["multi_date_count"], x["size"]), reverse=True
)

print(
    f"Found {len(scored_clusters)} clusters with at least 6 intersections in raw data.\n"
)
print("TOP 15 CLUSTERS (sorted by calibration/validation coverage):\n")

for idx, sc in enumerate(scored_clusters[:15]):
    print(f"{'=' * 75}")
    print(f"RANK #{idx + 1} | Cluster #{sc['cluster_num']} | Anchor: {sc['anchor']}")
    print(f"  Total intersections: {sc['size']}")
    print(
        f"  Found in raw data: {sc['found_in_raw']} | Missing: {sc['missing_from_raw']}"
    )
    print(f"  Intersections with 2+ dates: {sc['multi_date_count']}")
    print(
        f"  BEST CALIBRATION DATE: {sc['best_cal_date'].strftime('%Y-%m-%d') if sc['best_cal_date'] else 'NONE'}"
    )
    print(
        f"  BEST VALIDATION DATE:  {sc['best_val_date'].strftime('%Y-%m-%d') if sc['best_val_date'] else 'NONE'}"
    )
    print(f"  Intersections with BOTH dates: {sc['best_coverage']}")
    print(f"  Top shared dates (date -> #intersections):")
    for d, ids in sc["shared_dates"]:
        print(f"    {d.strftime('%Y-%m-%d')}: {len(ids)} intersections")
    print()

# Save full results
with open("data/cluster_scored.txt", "w") as f:
    for idx, sc in enumerate(scored_clusters):
        f.write(f"RANK #{idx + 1} | Cluster #{sc['cluster_num']} | {sc['anchor']}\n")
        f.write(
            f"  Coverage: {sc['best_coverage']} intersections with both cal + val dates\n"
        )
        f.write(f"  Cal: {sc['best_cal_date']} | Val: {sc['best_val_date']}\n")
        f.write(f"  Members:\n")
        for _, row in sc["members"].iterrows():
            cid = str(row["centreline_id"])
            ndates = dates_per_id.get(cid, 0)
            f.write(
                f"    - {row['location_name']} (ID: {cid}, {ndates} dates in raw)\n"
            )
        f.write("\n")

print(f"Saved full scored list to data/cluster_scored.txt")


Summary file: 6390 intersections
Raw file: 359315 rows, 4138 unique intersections

Built 412 clusters. Now checking raw data coverage...

Found 270 clusters with at least 6 intersections in raw data.

TOP 15 CLUSTERS (sorted by calibration/validation coverage):

RANK #1 | Cluster #185 | Anchor: King Georges Rd / Prince Edward Dr N (East)
  Total intersections: 12
  Found in raw data: 10 | Missing: 2
  Intersections with 2+ dates: 10
  BEST CALIBRATION DATE: 2024-10-23
  BEST VALIDATION DATE:  2024-04-30
  Intersections with BOTH dates: 9
  Top shared dates (date -> #intersections):
    2024-10-23: 10 intersections
    2024-04-30: 9 intersections
    2022-05-03: 5 intersections
    2023-06-20: 3 intersections
    2023-09-20: 1 intersections

RANK #2 | Cluster #45 | Anchor: Front St E / Scott St / Berczy Park Trl
  Total intersections: 12
  Found in raw data: 12 | Missing: 0
  Intersections with 2+ dates: 9
  BEST CALIBRATION DATE: 2020-01-16
  BEST VALIDATION DATE:  2020-01-25
  Interse

In [7]:
import pandas as pd

raw = pd.read_csv("data/tmc_raw_data_2020_2029.csv")
raw["centreline_id"] = raw["centreline_id"].astype(str)
raw["count_date"] = pd.to_datetime(raw["count_date"])

# All 12 IDs from Cluster #45 — REPLACE with your actual IDs
my_ids = [
    "13467132",
    "13467037",
    "13467239",
    "13467327",
    "13467080",
    "14254942",
    "13467425",
    "13467160",
    "13466814",
    "13466851",
    "13466729",
    "1146259",
]

# Find which IDs have BOTH dates
cal_date = "2020-01-16"
val_date = "2020-01-25"

for cid in my_ids:
    dates = raw[raw["centreline_id"] == cid]["count_date"].dt.date.unique()
    has_cal = pd.to_datetime(cal_date).date() in dates
    has_val = pd.to_datetime(val_date).date() in dates
    name = (
        raw[raw["centreline_id"] == cid]["location_name"].iloc[0]
        if len(raw[raw["centreline_id"] == cid]) > 0
        else "NOT FOUND"
    )
    status = (
        "✓ BOTH"
        if (has_cal and has_val)
        else ("✓ CAL only" if has_cal else ("✓ VAL only" if has_val else "✗ NONE"))
    )
    print(f"{status} | {name} (ID: {cid})")


✓ BOTH | Front St E / Scott St / Berczy Park Trl (ID: 13467132)
✓ BOTH | Wellington St E / Scott St / Berczy Park Trl (ID: 13467037)
✓ BOTH | Yonge St / Front St E / Front St W (ID: 13467239)
✓ BOTH | The Esplanade / Scott St (ID: 13467327)
✓ BOTH | Yonge St / Wellington St E / Wellington St W (ID: 13467080)
✓ BOTH | Front St E / Church St (ID: 14254942)
✓ BOTH | Yonge St / The Esplanade (ID: 13467425)
✓ BOTH | The Esplanade / Church St (ID: 13467160)
✗ NONE | King St E / Victoria St (ID: 13466814)
✗ NONE | King St E / Yonge St / King St W (ID: 13466851)
✗ NONE | King St E / Church St (ID: 13466729)
✗ NONE | Yonge St: Gardiner W / Yonge N Ramp - The Esplanade (ID: 1146259)


In [8]:
ary = pd.read_csv("data/tmc_most_recent_summary_data.csv")
summary["centreline_id"] = summary["centreline_id"].astype(str)

# Your 8 confirmed IDs
my_ids = [
    "13467132",
    "13467037",
    "13467239",
    "13467327",
    "13467080",
    "14254942",
    "13467425",
    "13467160",
]

subset = summary[summary["centreline_id"].isin(my_ids)]

print("Your 8 intersections:")
print(
    subset[["location_name", "centreline_id", "latitude", "longitude"]].to_string(
        index=False
    )
)

# Bounding box with 0.005 degree buffer (~500m)
min_lat = subset["latitude"].min() - 0.005
max_lat = subset["latitude"].max() + 0.005
min_lon = subset["longitude"].min() - 0.005
max_lon = subset["longitude"].max() + 0.005

print(f"\nBounding box for OSM export:")
print(f"  min_lat = {min_lat:.6f}")
print(f"  max_lat = {max_lat:.6f}")
print(f"  min_lon = {min_lon:.6f}")
print(f"  max_lon = {max_lon:.6f}")

print(f"\nOSM export URL (click this):")
print(
    f"https://www.openstreetmap.org/export#map=16/{(min_lat + max_lat) / 2:.4f}/{(min_lon + max_lon) / 2:.4f}"
)


Your 8 intersections:
                               location_name centreline_id  latitude  longitude
     Front St E / Scott St / Berczy Park Trl      13467132 43.647461 -79.375764
Yonge St / Wellington St E / Wellington St W      13467080 43.647832 -79.377337
          Yonge St / Front St E / Front St W      13467239 43.646885 -79.376974
                    Yonge St / The Esplanade      13467425 43.645784 -79.376503
                    The Esplanade / Scott St      13467327 43.646323 -79.375264
                   The Esplanade / Church St      13467160 43.647256 -79.373281
                      Front St E / Church St      14254942 43.648442 -79.373787
Wellington St E / Scott St / Berczy Park Trl      13467037 43.648129 -79.376040

Bounding box for OSM export:
  min_lat = 43.640784
  max_lat = 43.653442
  min_lon = -79.382337
  max_lon = -79.368281

OSM export URL (click this):
https://www.openstreetmap.org/export#map=16/43.6471/-79.3753


In [1]:
from pathlib import Path

NETWORK_DIR = Path(r"C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network")

files = list(NETWORK_DIR.iterdir())

for file in files:
    print(file)


C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\build.bat
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\edgeData.xml
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.dat.add.xml
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.net.xml.gz
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.netccfg
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.netecfg
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.ogcfg
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.passenger.rou.xml
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.passenger.trips.xml
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.poly.xml.gz
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.polycfg
C:\Users\danielshola\Desktop\traffic-digital-twin\toronto\network\osm.sumocfg
C:\Users\danielshola\Desktop\traff